# Initializing the environment

In [ ]:
from gymnasium import make
from samples.llm_interface import SoloGPT4Interfacer, MultiGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
from experiment_object import Experiment
os.environ["OPENAI_API_KEY"] = ""

n_player1 = 2
n_player2 = 2
# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
all_classes = ['halfling_rogue.yml', 'high_elf_fighter.yml']
all_players = ["Alysha", "Bernard", "Cedric", "Didier", "Eric", "Francois", "Gertrude", "Heloise", "Isabelle"]
players = random.sample(all_players, n_player1 + n_player2)
a_player = [(random.choice(all_classes), player) for player in players[:n_player1]]
e_player = [(random.choice(all_classes), player) for player in players[n_player1:]]

conversational_groups = {"a":True, "b":False}
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=a_player,
    enemies=e_player,
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000))



agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agent_type = MultiGPT4Interfacer if conversational_groups[gr] else SoloGPT4Interfacer
    agents[character.name] = (agent_type(debug=False, explain=True, name=character.name), gr, character)

agents


{'seed': 29, 'options': None}
Gertrude rolled initiative d20(12) + 5 value 17.2
Francois rolled initiative d20(13) + 5 value 18.2
Didier rolled initiative d20(14) + 5 value 19.2
Eric rolled initiative d20(1) + 5 value 6.2
Gertrude rolled initiative d20(2) + 5 value 7.2
Francois rolled initiative d20(4) + 5 value 9.2
Didier rolled initiative d20(15) + 5 value 20.2
Eric rolled initiative d20(8) + 5 value 13.2
Combat begins with 4 players.
Players: <p>Gertrude (rogue-2) Team a</p>
<p>Francois (fighter-2) Team a</p>
<p>Didier (fighter-2) Team b</p>
<p>Eric (fighter-2) Team b</p>
======== Didier starts their turn. ========
======== Didier starts their turn. ========


{'Didier': (<samples.llm_interface.SoloGPT4Interfacer at 0x7fd3577acc00>,
  'b',
  Didier),
 'Eric': (<samples.llm_interface.SoloGPT4Interfacer at 0x7fd3577ad040>,
  'b',
  Eric),
 'Francois': (<samples.llm_interface.MultiGPT4Interfacer at 0x7fd3577acf30>,
  'a',
  Francois),
 'Gertrude': (<samples.llm_interface.MultiGPT4Interfacer at 0x7fd3577ad150>,
  'a',
  Gertrude)}

In [3]:
expe = Experiment(env, env.env.env, agents, conversational_groups=conversational_groups, debug=False)
expe.run_till_end(max_step=20)

Eric attacked Heloise with Longbow and hits with attack roll d20(12) + 7 = 19.
Heloise took d8(8) + 5 = 13 piercing damage. 


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(


Eric uses second wind to recover d10(5) + 2=7 hit points.
======== Heloise starts their turn. ========
======== Heloise starts their turn. ========
Heloise uses second wind to recover d10(2) + 2=4 hit points.
Heloise dodges.
======== Gertrude starts their turn. ========
======== Gertrude starts their turn. ========
Gertrude attacked Isabelle with Longbow and hits with attack roll d20(16) + 7 = 23.
Isabelle took d8(8) + 5 = 13 piercing damage. 
Gertrude uses second wind to recover d10(7) + 2=9 hit points.
======== Isabelle starts their turn. ========
======== Isabelle starts their turn. ========
Isabelle tried to attack Heloise with disadvantage['target_dodge'] with Shortbow but missed with d20(3 | 1*) + 7 lucky -> d20(3* | 17) + 7= 10.
Isabelle disengages.
Isabelle moved to [4, 2] 5 feet
Isabelle moved to [3, 2] 5 feet
Isabelle moved to [2, 2] 5 feet
Isabelle moved to [1, 2] 5 feet
======== Eric starts their turn. ========
======== Eric starts their turn. ========
Eric attacked Heloise

True

In [4]:
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


expe.backlog
metrics = combat_metrics(expe.dnd_environment, agents)
print(metrics)
score = combat_score(metrics)
print(score)

{'win': False, 'turns_taken': 20, 'survivors': {'Eric': (24, 24), 'Isabelle': (3, 16)}}
-8.125


In [ ]:
metrics

{'win': False,
 'turns_taken': 115,
 'survivors': {'Francois': (0, 24),
  'Alysha': (0, 16),
  'Eric': (5, 24),
  'Bernard': (24, 24)}}

In [ ]:
expe.conversations

[[('Alysha',
   "Francois, I'm badly hurt and staying hidden for now. Focus on keeping Eric and Bernard busy—I'll look for a moment to strike safely. Try to draw their attention if you can!"),
  ('Francois',
   "Understood, Alysha! I'll keep the pressure on Eric and try to draw both enemies' attention. Stay safe—strike whenever you see an opening. If they split up, I'll call it out so we can take advantage. Let's keep each other posted."),
  (None, 'Conversation closed')],
 [('Francois',
   "Alysha, Eric is badly hurt but is Dodging right now, making him harder to hit. Stay safe and ready—we'll finish him as soon as an opportunity opens up. Once Eric is down, we both shift our focus to Bernard. Let me know if you need support or see a better move."),
  ('Alysha',
   'Thanks, Francois. I’m keeping my head down for now—no openings yet, but I’m watching for any slip in Eric’s defense. If he drops his guard, I’ll move in for a Sneak Attack. If things get dicey or you need a distraction, gi